# PCA model inversion — food texture worked example

This notebook accompanies the *Applications of Latent Variable Models* section of [Process Improvement using Data](https://learnche.org/pid). It builds a two-component PCA model of the food-texture data and then **inverts** it: starting from a desired score, it recovers the pastry recipe that would land an observation there.

The final figure is interactive — hover (or tap) anywhere in the score plot and the tooltip shows the inverted recipe for that target.

Requirements: `pip install process-improve numpy pandas plotly`

## 1. Load and preprocess the data

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from process_improve.multivariate import PCA, MCUVScaler

food = pd.read_csv("https://openmv.net/file/food-texture.csv")
scaler = MCUVScaler().fit(food)
food_mcuv = scaler.transform(food)
food.head()

## 2. Fit a two-component PCA model

In [ ]:
A = 2
model = PCA(n_components=A).fit(food_mcuv)
print(model.r2_cumulative_)

## 3. Check the model

Before using the model to *generate* recipes, confirm it is a faithful summary of the data: the SPE and Hotelling's $T^2$ of each pastry, with their 95% limits.

In [ ]:
spe = model.spe_.iloc[:, -1]
t2 = model.hotellings_t2_.iloc[:, -1]
spe_limit = float(model.spe_limit(conf_level=0.95))
t2_limit = float(model.hotellings_t2_limit(conf_level=0.95))

fig = go.Figure()
fig.add_trace(go.Scatter(y=spe.values, mode="lines+markers", name="SPE"))
fig.add_hline(y=spe_limit, line_color="red", line_dash="dash",
              annotation_text="95% limit")
fig.update_layout(title="SPE per pastry", xaxis_title="Pastry number",
                  yaxis_title="SPE", height=320)
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(y=t2.values, mode="lines+markers", name="T2"))
fig.add_hline(y=t2_limit, line_color="red", line_dash="dash",
              annotation_text="95% limit")
fig.update_layout(title="Hotelling's T-squared per pastry",
                  xaxis_title="Pastry number", yaxis_title="T-squared",
                  height=320)
fig.show()

## 4. Score and loadings plots

In [ ]:
scores = model.scores_
ci_x, ci_y = model.ellipse_coordinates(score_horiz=1, score_vert=2,
                                       conf_level=0.95)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Scores", "Loadings"))
fig.add_trace(go.Scatter(x=scores.iloc[:, 0], y=scores.iloc[:, 1],
                         mode="markers", marker=dict(color="black"),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=ci_x, y=ci_y, mode="lines",
                         line=dict(color="palevioletred"),
                         showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=model.loadings_.iloc[:, 0],
                         y=model.loadings_.iloc[:, 1], mode="markers+text",
                         text=model.loadings_.index,
                         textposition="bottom center",
                         showlegend=False), row=1, col=2)
fig.update_xaxes(title_text="t1", row=1, col=1)
fig.update_yaxes(title_text="t2", row=1, col=1)
fig.update_xaxes(title_text="p1", row=1, col=2)
fig.update_yaxes(title_text="p2", row=1, col=2)
fig.show()

## 5. Invert the model for a single target

With the loadings matrix $\mathbf{P}$ and a target score $\mathbf{t} = [t_1, t_2]$, the scaled variables are $\mathbf{x}_\text{scaled} = \mathbf{t}\,\mathbf{P}^{T}$; the scaler then undoes the centering and scaling.

In [ ]:
P = model.loadings_.values

def invert(t1, t2):
    """Return the pastry recipe for a target score (t1, t2)."""
    x_scaled = np.array([[t1, t2]]) @ P.T
    recipe = scaler.inverse_transform(pd.DataFrame(x_scaled, columns=food.columns))
    return recipe.iloc[0]

print(invert(0, 0))     # the origin returns the average pastry
print(invert(2, -1))    # a point in the profitable quadrant

## 6. Interactive inversion across the whole score plot

The trick: lay a high-resolution heatmap behind the scores, make it fully transparent, and precompute the inverted recipe for every grid cell. Plotly's native hover tooltip then acts as a live inset — no callbacks, works on desktop hover and mobile tap alike.

In [ ]:
pad = 0.5
t1_range = (float(min(scores.iloc[:, 0].min(), ci_x.min())) - pad,
            float(max(scores.iloc[:, 0].max(), ci_x.max())) + pad)
t2_range = (float(min(scores.iloc[:, 1].min(), ci_y.min())) - pad,
            float(max(scores.iloc[:, 1].max(), ci_y.max())) + pad)

GRID = 60
t1_grid = np.linspace(t1_range[0], t1_range[1], GRID)
t2_grid = np.linspace(t2_range[0], t2_range[1], GRID)
TT1, TT2 = np.meshgrid(t1_grid, t2_grid)
T_flat = np.column_stack([TT1.ravel(), TT2.ravel()])

# Vectorised inversion of the whole grid.
X_orig = scaler.inverse_transform(
    pd.DataFrame(T_flat @ P.T, columns=food.columns)).values

hover = np.empty(T_flat.shape[0], dtype=object)
for idx in range(T_flat.shape[0]):
    t1, t2 = T_flat[idx]
    lines = [f"<b>Target</b>  t1={t1:.2f}  t2={t2:.2f}", "<b>Inverted recipe</b>"]
    for name, v in zip(food.columns, X_orig[idx]):
        lines.append(f"  {name}: {v:.2f}")
    hover[idx] = "<br>".join(lines)
hover = hover.reshape((GRID, GRID))

loadings_scale = 0.9 * min(scores.iloc[:, 0].abs().max(),
                           scores.iloc[:, 1].abs().max())

fig = go.Figure()
fig.add_trace(go.Heatmap(
    x=t1_grid, y=t2_grid, z=np.zeros((GRID, GRID)),
    text=hover, hoverinfo="text",
    colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],
    showscale=False, name=""))
fig.add_trace(go.Scatter(x=scores.iloc[:, 0], y=scores.iloc[:, 1],
    mode="markers", marker=dict(size=7, color="black"),
    name="Observed pastries", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=ci_x, y=ci_y, mode="lines",
    line=dict(color="palevioletred", width=2),
    name="95% T-squared ellipse", hoverinfo="skip"))
fig.add_trace(go.Scatter(
    x=P[:, 0] * loadings_scale, y=P[:, 1] * loadings_scale,
    mode="markers+text", marker=dict(size=12, symbol="x", color="lightskyblue"),
    text=list(food.columns), textposition="bottom center",
    name="Loadings", hoverinfo="skip"))
fig.update_layout(
    title="PCA model inversion: hover anywhere to see the inverted recipe",
    xaxis_title="t1", yaxis_title="t2",
    xaxis=dict(range=t1_range),
    yaxis=dict(range=t2_range, scaleanchor="x", scaleratio=1),
    hovermode="closest", height=560, margin=dict(l=70, r=20, t=70, b=50))
fig.show()

## Notes

* A target is only realistic if it lies **within** the model — inside the $T^2$ ellipse and implying a low SPE. A target far outside the cloud of pastries inverts to a recipe the process has never been shown to produce.
* The inversion here is **unconstrained**: all five variables move freely. If some variables are fixed (a hardness specification, say), the constrained inversion projects onto the remaining free subspace.
* Further reading: Jaeckle and MacGregor, *Product design through multivariate statistical analysis of process data*, AIChE Journal, 44, 1105-1118, 1998.